In [1]:
# ==========================================================
# ACTIVIDAD 13. Detalle de los saltos extremos
# Lista cada salto extremo o revertido de las 1.593 empresas
# elegibles, con su contexto, para clasificarlo como error de
# datos o como evento real. No modifica ni excluye nada.
# ==========================================================
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive

SALTO = 0.4                        # |r| > 0,4 para considerar una posible reversión
REVERSION = 0.1                    # el día siguiente deshace el salto casi por completo
EXTREMO = np.log(2)                # |r| > 0,693: el precio se duplica o se reduce a la mitad
FIN_ENTRENAMIENTO = pd.Timestamp("2022-12-30")
FIN_VALIDACION = pd.Timestamp("2024-12-31")

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
CARPETA_DATOS = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/01_datos")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Empresas elegibles y sus precios
# ----------------------------------------------------------
elegibles = pd.read_csv(CARPETA / "elegibles_muestra.csv", dtype={"ticker": str, "cik": str},
                        keep_default_na=False, na_values=[""]).set_index("ticker")
precios = pd.read_parquet(CARPETA_DATOS / "precios_universo.parquet",
                          columns=["ticker", "date", "close", "adj_close", "volume"])
precios = precios[precios["ticker"].isin(elegibles.index)].copy()
precios["ticker"] = precios["ticker"].astype(str)
precios = precios.sort_values(["ticker", "date"]).reset_index(drop=True)
print("Empresas:", precios["ticker"].nunique(), "| filas:", len(precios))

# ----------------------------------------------------------
# 2. Rendimientos y contexto de cada día
# ----------------------------------------------------------
t = precios["ticker"]
precios["r"] = np.log(precios["adj_close"]).groupby(t).diff()
precios["r_siguiente"] = precios["r"].groupby(t).shift(-1)
precios["precio_antes"] = precios["adj_close"].groupby(t).shift(1)
precios["volumen_antes"] = precios["volume"].groupby(t).shift(1).astype("Int64")

# Días previos con el precio congelado (r = 0) en las 5 sesiones anteriores.
# Un precio congelado antes de un gran salto sugiere que no había negociación real.
plano = (precios["r"] == 0).astype(int)
acumulado = plano.groupby(t).cumsum()
precios["planos_previos"] = acumulado.groupby(t).shift(1) - acumulado.groupby(t).shift(6)

precios["revertido"] = ((precios["r"].abs() > SALTO)
                        & (precios["r"] * precios["r_siguiente"] < 0)
                        & ((precios["r"] + precios["r_siguiente"]).abs() < REVERSION))
# El día siguiente a un salto revertido es su "recuperación", no un evento nuevo
precios["recuperacion"] = precios["revertido"].groupby(t).shift(1, fill_value=False)
precios["extremo"] = precios["r"].abs() > EXTREMO
precios["bloque"] = np.select([precios["date"] <= FIN_ENTRENAMIENTO, precios["date"] <= FIN_VALIDACION],
                              ["1 entrenamiento", "2 validacion"], default="3 prueba")

# ----------------------------------------------------------
# 3. Tabla de eventos (un evento = un día con salto)
# ----------------------------------------------------------
eventos = precios[precios["extremo"] | precios["revertido"]].copy()
eventos["tipo"] = np.select([eventos["revertido"], eventos["recuperacion"]],
                           ["revertido", "recuperacion"], default="extremo")
eventos["veces"] = np.exp(eventos["r"])              # factor de cambio del precio ese día
eventos = eventos[["ticker", "date", "bloque", "tipo", "precio_antes", "adj_close", "veces", "r",
                   "r_siguiente", "volumen_antes", "volume", "planos_previos"]]

print("\n=== EVENTOS POR BLOQUE Y TIPO (días | empresas) ===")
resumen = eventos.groupby(["bloque", "tipo"]).agg(dias=("date", "size"), empresas=("ticker", "nunique"))
print(resumen)
print("Total:", len(eventos), "días en", eventos["ticker"].nunique(), "empresas")

print("\n=== SALTOS REVERTIDOS (posibles errores aislados) ===")
print(eventos[eventos["tipo"] != "extremo"].to_string(index=False))

print("\n=== SALTOS TRAS UN PRECIO CONGELADO (3 o más días con r = 0 justo antes) ===")
print(eventos[eventos["planos_previos"] >= 3].to_string(index=False))

print("\n=== EMPRESAS CON MÁS DÍAS DE SALTO ===")
print(eventos.loc[eventos["tipo"] != "recuperacion", "ticker"].value_counts().head(15))

# ----------------------------------------------------------
# 4. Guardar para la revisión caso por caso
# ----------------------------------------------------------
eventos.to_csv(CARPETA / "saltos_detalle.csv", index=False)
print("\nGuardado: saltos_detalle.csv -", len(eventos), "eventos")

Mounted at /content/drive
Empresas: 1593 | filas: 2730402

=== EVENTOS POR BLOQUE Y TIPO (días | empresas) ===
                           dias  empresas
bloque          tipo                     
1 entrenamiento extremo      71        50
                revertido    13        12
2 validacion    extremo      20        19
                revertido     1         1
3 prueba        extremo      34        24
                revertido     1         1
Total: 140 días en 89 empresas

=== SALTOS REVERTIDOS (posibles errores aislados) ===
ticker       date          bloque      tipo  precio_antes  adj_close  veces      r  r_siguiente  volumen_antes    volume  planos_previos
  ACRE 2020-03-18 1 entrenamiento revertido         2.811      1.812  0.645 -0.439        0.340        1263300   1583000           0.000
  AROC 2020-03-18 1 entrenamiento revertido         2.641      1.467  0.556 -0.588        0.508        1393300   2112300           0.000
  ATER 2021-09-13 1 entrenamiento revertido       141.60